# Milestone 3: Model Comparison

This notebook compares three regression models:
- Median dummy regressor: a simple reference prediction.
- Linear regression: learns a weighted relationship between inputs
  and rental price.
- Decision tree regressor: learns rules that can capture nonlinear
  relationships.

## 1. Load and select the data

Only monthly USD listings are retained, matching Milestone 2.
Previously flagged suspicious records remain unchanged for this
initial comparison.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../data/apartments_for_rent_classified_10K.csv",
    sep=";",
    encoding="latin1"
)

data = df.loc[
    (df["price_type"] == "Monthly") &
    (df["currency"] == "USD")
].copy()

print("Original listings:", len(df))
print("Retained listings:", len(data))

Original listings: 10000
Retained listings: 9998


## 2. Input features and target

Eight inputs as decided in milestone 2:

In [2]:
numeric_features = [
    "bedrooms",
    "bathrooms",
    "square_feet",
    "latitude",
    "longitude"
]

categorical_features = [
    "cityname",
    "state",
    "has_photo"
]

features = numeric_features + categorical_features

X = data[features].copy()
y = data["price"].copy()

print("Input table shape:", X.shape)
print("Target shape:", y.shape)

Input table shape: (9998, 8)
Target shape: (9998,)


## 3. Evaluation of design

The same random 80/20 split from Milestone 2 is used, with a fixed seed of 42. All three models will be compared on the same five cross-validation folds using development data only. The final test set remains unused.
Possible repeated property listings are still a limitation of the random split.

In [3]:
from sklearn.model_selection import train_test_split, KFold

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Development listings:", len(X_dev))
print("Protected test listings:", len(X_test))

Development listings: 7998
Protected test listings: 2000


## 4. Preprocessing

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# will fill missing numerical values with the median for that column learned from the training data
numeric_pipeline = SimpleImputer(strategy="median")

# this pipeline fills missing values with 'unknown' and then uses hot encoder to turn categories like city or state into indicators
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# this will send each set of columns through the right steps and will combine results
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 5. Model for comparison

In [9]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# dummy will predict the training median rent
# linear regression will learn a weighted combination of features
# decision tree learns rules that split listings into groups
models = {
    "Median dummy": DummyRegressor(strategy="median"),
    "Linear regression": LinearRegression(),
    "Decision tree": DecisionTreeRegressor(
        max_depth=5,
        min_samples_leaf=20,
        random_state=42
    )
}

model_pipelines = {
    name: Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])
    for name, model in models.items()
}

print("Models:", list(model_pipelines))

Models: ['Median dummy', 'Linear regression', 'Decision tree']


In [7]:
from sklearn.model_selection import cross_validate

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error"
}

results = []

for name, pipeline in model_pipelines.items():
    scores = cross_validate(
        pipeline,
        X_dev,
        y_dev,
        cv=cv,
        scoring=scoring,
        error_score="raise"
    )

    mae = -scores["test_MAE"]
    rmse = -scores["test_RMSE"]

    results.append({
        "Model": name,
        "Mean MAE": mae.mean(),
        "MAE std": mae.std(ddof=1),
        "Mean RMSE": rmse.mean(),
        "RMSE std": rmse.std(ddof=1)
    })

print(f"Evaluated {len(results)} models.")

Evaluated 3 models.


In [8]:
comparison = pd.DataFrame(results).sort_values("Mean MAE")
display(comparison.round(2))

,Model,Mean MAE,MAE std,Mean RMSE,RMSE std
1,Linear regression,328.57,17.51,732.64,363.73
2,Decision tree,395.13,18.63,835.43,326.61
0,Median dummy,556.46,32.80,1115.79,310.15


### Initial model comparison

Linear regression performed best in five fold cross-validation, with a mean MAE of 328.57 dollars and a mean RMSE of 732.64 dollars. Its mean MAE was about 41% lower than the median dummy baseline. The decision tree also improved on the baseline, but its initial settings produced a higher mean MAE and RMSE than linear regression.

These results are from the development folds.
The protected test set was not used